# 23 — Synthetic Historical Control on the monthly latent panel with a learned covariate mapping

Plan of 10 September 2026 (`quirky-growing-wombat`). Model y_t = g(x_t) + ℓ_t + δ_t d_t + ε_t: the covariate term
x_t'β of the paper (slide 6, steps 1–3) is replaced by g, a multi-output ridge regression of the 980-d latent on
land-cover class, elevation and slope, fitted **across sites at each period** on untreated rows only (every site at
P01–P10, control sites at P11–P21), leave-one-site-out so no site's own latent enters its own g. Everything after
that — kernel stage, blocks, stepwise selection, convex weights, counterfactual, effect — is `panel_shc.py`
unchanged, run on u = y − g by `panel_shc_run.py --covariate site`. The temporal validation (fit P01–P08,
predict P09–P10) and the effect are scored on the observed latent y against g + ℓ̂⁽⁰⁾, so every number has the
same definition and units as in notebook 22 / report §5–§7.

**Left = covariate-free run of 4 September (existing CSVs, not recomputed); right = the run with g.** S1 before S2.
Every number is read from the CSVs.

In [1]:
import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 80)
def load(suffix):
    F = pd.read_csv(f"panel_shc_fits{suffix}.csv"); E = pd.read_csv(f"panel_shc_effects{suffix}.csv")
    S = pd.read_csv(f"panel_shc_summary{suffix}.csv"); return F, E, S
F0, E0, S0 = load(""); F1, E1, S1 = load("_covariate"); G = pd.read_csv("panel_covariate.csv")
print(len(F0), "fits without g;", len(F1), "fits with g; same cells:", set(map(tuple, F0[["fill","sensor","site","n","m"]].values)) == set(map(tuple, F1[["fill","sensor","site","n","m"]].values)))
KEYS = ["fill", "sensor", "n", "m"]
def side(S_a, S_b, cols, label_a="without g", label_b="with g"):
    a = S_a.set_index(KEYS)[cols].add_suffix(f" | {label_a}"); b = S_b.set_index(KEYS)[cols].add_suffix(f" | {label_b}")
    out = a.join(b, how="outer")
    return out[[c for pair in zip(a.columns, b.columns) for c in pair]].reset_index()

7722 fits without g; 7722 fits with g; same cells: True


## 0. The mapping g: penalty, leave-one-site-out R² per channel and period, coefficients at P09–P12

R² is the share of the cross-site variance of the latent at that period explained by the descriptors, leave-one-site-out, averaged over the 196 parcels of each channel. Coefficients are per feature (reference class 41; elevation and slope z-scored over the 286 sites), averaged over the parcels of each channel.

In [2]:
r2 = G[G.kind == "r2"]
print("penalty chosen per fill x sensor (leave-one-site-out squared error over P01-P10):")
print(r2.drop_duplicates(["fill","sensor"])[["fill","sensor","lam","lam_flag"]].to_string(index=False))
for sensor in ("sentinel1", "sentinel2"):
    print(f"\n=== {sensor}: leave-one-site-out R^2 per channel by period ===")
    print(r2[r2.sensor == sensor][["fill","period","n_rows","n_leverage_one"] + [f"r2_ch{c}" for c in range(1,6)]].to_string(index=False, float_format=lambda v: f"{v:.3f}"))
cf = G[G.kind == "coef"]
for sensor in ("sentinel1", "sentinel2"):
    print(f"\n=== {sensor}, chip-mean fill: coefficients per channel (mean over parcels) ===")
    print(cf[(cf.sensor == sensor) & (cf.fill == "chipmean")][["period","feature"] + [f"coef_ch{c}" for c in range(1,6)]].to_string(index=False, float_format=lambda v: f"{v:.4f}"))

penalty chosen per fill x sensor (leave-one-site-out squared error over P01-P10):
    fill    sensor    lam lam_flag
chipmean sentinel1 1000.0 interior
chipmean sentinel2  100.0 interior
histfill sentinel1 1000.0 interior
histfill sentinel2  100.0 interior

=== sentinel1: leave-one-site-out R^2 per channel by period ===
    fill period  n_rows  n_leverage_one  r2_ch1  r2_ch2  r2_ch3  r2_ch4  r2_ch5
chipmean    P01      71           0.000  -0.029  -0.028  -0.028  -0.029  -0.029
chipmean    P02     132           0.000  -0.015  -0.015  -0.015  -0.015  -0.016
chipmean    P03     132           0.000  -0.015  -0.015  -0.015  -0.015  -0.015
chipmean    P04     132           0.000  -0.015  -0.015  -0.015  -0.015  -0.015
chipmean    P05     132           0.000  -0.015  -0.015  -0.015  -0.015  -0.016
chipmean    P06     132           0.000  -0.015  -0.014  -0.015  -0.015  -0.016
chipmean    P07     132           0.000  -0.015  -0.015  -0.015  -0.015  -0.016
chipmean    P08     132           0.00

## 1. Stage 1 — bandwidths chosen by leave-one-out CV (treated sites, effect window P01–P10)

In [3]:
def bw(F):
    T = F[F.group == "treatment"]; hh = T[T.m == T.m.min()][["fill","sensor","site","h","h_flag"]].drop_duplicates(["fill","sensor","site"])
    return hh.groupby(["fill","sensor"]).agg(sites=("site","nunique"), h_median=("h","median"), h_min=("h","min"), h_max=("h","max"),
                                            interior=("h_flag", lambda v: int((v=="interior").sum())))
print(bw(F0).join(bw(F1), lsuffix=" | without g", rsuffix=" | with g").to_string())

                    sites | without g  h_median | without g  h_min | without g  h_max | without g  interior | without g  sites | with g  h_median | with g  h_min | with g  h_max | with g  interior | with g
fill     sensor                                                                                                                                                                                              
chipmean sentinel1                 12              0.712643           0.501187           1.995262                    12              12           0.712643        0.501187        1.995262                 12
         sentinel2                 15              0.501187           0.158489           1.258925                    15              15           0.501187        0.316228        1.000000                 15
histfill sentinel1                 12              0.712643           0.501187           1.995262                    12              12           0.712643        0.501187      

## 2. Temporal validation (fit on P01–P(10−n), predict the last n pre-hurricane months): RMSE over the 980 coordinates between the observed latent and the counterfactual, mean over treated sites

Reference = the site's own P01..P(10−n) mean of the observed latent (same in both runs).

In [4]:
cols = ["placebo_rmse", "placebo_rmse_raw", "placebo_rmse_ownmean", "placebo_beats_ownmean", "most_recent_only", "n_eff_mean"]
for sensor in ("sentinel1", "sentinel2"):
    t = side(S0[S0.sensor == sensor], S1[S1.sensor == sensor], cols)
    print(f"\n=== {sensor} ===")
    print(t.sort_values(["fill","n","m"]).to_string(index=False, float_format=lambda v: f"{v:.4f}"))


=== sentinel1 ===
    fill    sensor  n  m  placebo_rmse | without g  placebo_rmse | with g  placebo_rmse_raw | without g  placebo_rmse_raw | with g  placebo_rmse_ownmean | without g  placebo_rmse_ownmean | with g  placebo_beats_ownmean | without g  placebo_beats_ownmean | with g  most_recent_only | without g  most_recent_only | with g  n_eff_mean | without g  n_eff_mean | with g
chipmean sentinel1  2  2                    0.2303                 0.2310                        0.2666                     0.2677                            0.2360                         0.2360                            11.0000                         10.0000                            12                         12                  1.0000               1.0000
chipmean sentinel1  2  3                    0.2303                 0.2310                        0.2666                     0.2677                            0.2360                         0.2360                            11.0000                     

## 3. Pre-period matching and donor selection (the series matched is y without g, u = y − g with g)

In [5]:
cols = ["N", "mse_pre_mean", "passes_nu", "most_recent_only", "oldest_only", "n_selected_mean", "n_eff_mean"]
for sensor in ("sentinel1", "sentinel2"):
    t = side(S0[S0.sensor == sensor], S1[S1.sensor == sensor], cols)
    print(f"\n=== {sensor} ===")
    print(t.sort_values(["fill","n","m"]).to_string(index=False, float_format=lambda v: f"{v:.5f}"))


=== sentinel1 ===
    fill    sensor  n  m  N | without g  N | with g  mse_pre_mean | without g  mse_pre_mean | with g  passes_nu | without g  passes_nu | with g  most_recent_only | without g  most_recent_only | with g  oldest_only | without g  oldest_only | with g  n_selected_mean | without g  n_selected_mean | with g  n_eff_mean | without g  n_eff_mean | with g
chipmean sentinel1  2  2              7           7                   0.00431                0.00429                      0                   0                            12                         12                        0                     0                      1.00000                   1.00000                 1.00000              1.00000
chipmean sentinel1  2  3              6           6                   0.00418                0.00416                      0                   0                            12                         12                        0                     0                      1.00000         

## 4. Effect run: ‖δ̂‖/√980 per post period, treated sites against their matched control sites (m = 4)

In [6]:
def eff(E):
    g = E[E.m == 4]
    return g.groupby(["sensor","fill","n","post_period","group"]).agg(sites=("site","nunique"), delta_rms=("delta_rms","mean"), delta_raw=("delta_raw_rms","mean"))
print(eff(E0).join(eff(E1), lsuffix=" | without g", rsuffix=" | with g").round(4).to_string())
cols = ["delta_rms_mean", "nc_sites", "nc_rank1", "nc_p_mean"]
for sensor in ("sentinel1", "sentinel2"):
    t = side(S0[(S0.sensor == sensor) & (S0.m == 4)], S1[(S1.sensor == sensor) & (S1.m == 4)], cols)
    print(f"\n=== {sensor}, m = 4: treated departure and negative-control rank ===")
    print(t.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

                                                 sites | without g  delta_rms | without g  delta_raw | without g  sites | with g  delta_rms | with g  delta_raw | with g
sensor    fill     n post_period group                                                                                                                                  
sentinel1 chipmean 2 P11         counterfactual                120                 0.2134                 0.2688             120              0.2142              0.2698
                                 treatment                      12                 0.2086                 0.2595              12              0.2090              0.2601
                     P12         counterfactual                120                 0.2638                 0.3025             120              0.2641              0.3030
                                 treatment                      12                 0.2648                 0.3007              12              0.2643       

## 5. Per treated site, n = 2, m = 4, historical fill: departure at P11 and P12, rank among the eleven, validation error (as report §7)

In [7]:
def site_table(F, fill="histfill", n=2, m=4):
    T = F[(F.group=="treatment")&(F.fill==fill)&(F.n==n)&(F.m==m)&(F.ok==True)]
    Cc = F[(F.group=="counterfactual")&(F.fill==fill)&(F.n==n)&(F.m==m)&(F.ok==True)]
    out = []
    for _, r in T.iterrows():
        c = Cc[Cc.matched_treatment == r.site]
        out.append({"sensor": r.sensor, "site": r.site[-4:], "h": r.h, "P11": r.get("delta_rms_k1", np.nan), "P12": r.get("delta_rms_k2", np.nan),
                    "controls_median": float(c.delta_rms_mean.median()) if len(c) else np.nan,
                    "rank": 1 + int((c.delta_rms_mean >= r.delta_rms_mean).sum()) if len(c) else np.nan,
                    "validation": r.get("placebo_rmse", np.nan), "reference": r.get("placebo_rmse_ownmean", np.nan)})
    return pd.DataFrame(out).set_index(["sensor","site"])
print(site_table(F0).join(site_table(F1), lsuffix=" | without g", rsuffix=" | with g").round(3).to_string())

                h | without g  P11 | without g  P12 | without g  controls_median | without g  rank | without g  validation | without g  reference | without g  h | with g  P11 | with g  P12 | with g  controls_median | with g  rank | with g  validation | with g  reference | with g
sensor    site                                                                                                                                                                                                                                                                         
sentinel1 0015          0.794            0.246            0.289                        0.218                 3                   0.253                  0.256       0.794         0.246         0.289                     0.218              3                0.254               0.256
          0016          0.501            0.211            0.278                        0.246                 6                   0.251                  0.249   

## 6. Per-channel mean δ̂ (treated vs controls, m = 4, n = 2, historical fill)

In [8]:
ch = [f"delta_ch{c}" for c in range(1,6)]
def chan(E):
    g = E[(E.m==4)&(E.n==2)&(E.fill=="histfill")]
    return g.groupby(["sensor","post_period","group"])[ch].mean()
print(chan(E0).join(chan(E1), lsuffix=" | without g", rsuffix=" | with g").round(4).to_string())

                                      delta_ch1 | without g  delta_ch2 | without g  delta_ch3 | without g  delta_ch4 | without g  delta_ch5 | without g  delta_ch1 | with g  delta_ch2 | with g  delta_ch3 | with g  delta_ch4 | with g  delta_ch5 | with g
sensor    post_period group                                                                                                                                                                                                                                
sentinel1 P11         counterfactual                -0.0088                 0.0079                 0.0096                 0.0184                -0.0035              0.0008              0.0000             -0.0013             -0.0010             -0.0007
                      treatment                     -0.0003                 0.0084                 0.0128                 0.0189                -0.0064              0.0086              0.0012              0.0022             -0.0001             